<a href="https://colab.research.google.com/github/christinengalle19-collab/Assignement2New/blob/main/Assignment11ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This project focuses on performing image classification through machine learning methods.

In [ ]:
import zipfile
import io
from google.colab import files
from pathlib import Path as path
import matplotlib.pyplot as plt
import numpy as np
from sklearn import svm, metrics, datasets
from sklearn.utils import Bunch
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from skimage.io import imread
from skimage.transform import resize
import skimage.io
import os

In [ ]:
def load_images_files(container_path, dimension=(64,64)):
    image_dir = path(container_path)
    folders = [directory for directory in image_dir.iterdir() if directory.is_dir()]
    categories = [fo.name for fo in folders]

    images, flat_data, target = [],[],[]
    for i, direc in enumerate(folders):
        for file in direc.iterdir():
            img = skimage.io.imread(file)
            img_resized = resize(img, dimension, anti_aliasing=True, mode='reflect')
            flat_data.append(img_resized.flatten())
            images.append(img_resized)
            target.append(i)
    flat_data = np.array(flat_data)
    target = np.array(target)
    images = np.array(images)

    return images, flat_data, target, categories

In [ ]:
#Load the dataset
uploaded = files.upload()

In [ ]:
import os

zip_ref = zipfile.ZipFile('images.zip', 'r')

# Create a directory to extract into, if it doesn't already exist
extraction_path = '/content/images/'
os.makedirs(extraction_path, exist_ok=True)

zip_ref.extractall(extraction_path)
zip_ref.close()

In [ ]:
# Load the dataset using the defined function
images, flat_data, target, categories = load_images_files(extraction_path + 'images')

# Create a Bunch object similar to scikit-learn's datasets
image_dataset = Bunch(data=flat_data, target=target, target_names=categories, images=images)

In [ ]:
from skimage.color import xyz_tristimulus_values
#Splitting the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split( image_dataset.data, image_dataset.target, test_size = 0.3,random_state =42)

In [ ]:
from sklearn.metrics import classification_report

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_clf = RandomForestClassifier(random_state=42)
rf_grid_search = GridSearchCV(estimator=rf_clf, param_grid=rf_param_grid, cv=5, n_jobs=-1, verbose=2)
rf_grid_search.fit(x_train, y_train)

y_pred = rf_grid_search.predict(x_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits


## I see that my model performs reasonably well overall with 77% accuracy, but some classes are clearly harder to predict than others. I notice strong performance for classes 2 and 3, while classes 1 and 4 show consistent confusion, especially with each other. The confusion matrix confirms these weaknesses and helps me understand where the model needs improvement.



In [ ]:
rf_grid_search.best_params_

#

In [ ]:
svm_pa_grid = {'C': [0.1, 1, 10], 'gamma': [1, 0.1, 0.01], 'kernel': ['rbf', 'poly']}

svm_clf = SVC()
svm_grid_search = GridSearchCV(estimator=svm_clf, param_grid=svm_pa_grid, cv=5, n_jobs=-1, verbose=2)
svm_grid_search.fit(x_train, y_train)

y_pred = svm_grid_search.predict(x_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


##I see that my model performs well overall with 82% accuracy, but the classes vary in difficulty. Classes 0, 1, and 2 show strong precision and recall, while class 3 remains challenging and class 4 is often predicted correctly but with lower precision. The confusion matrix confirms that classes 3 and 4 still generate misclassifications, helping me identify where the model needs further refinement.

In [ ]:
#Displays the best parameter
print("Best RF parameters:", rf_grid_search.best_params_)
print("Best SVM parameters:", svm_grid_search.best_params_)

In [ ]:
#Train the Random Forest model on the training data.
rf_clf = RandomForestClassifier(n_estimators=300, random_state=42)
rf_clf.fit(x_train, y_train)


In [ ]:
#Model Evaluation:
#Make predictions on the test set using the best model.
y_pred = rf_clf.predict(x_test)

#Evaluate the model using metrics such as accuracy, precision, recall, and F1-score.
print(classification_report(y_test, y_pred))

#Create a confusion matrix and classification report using confusion_matrix and classification_report from sklearn.metrics.
print(confusion_matrix(y_test, y_pred))

#Visualize the confusion matrix using matplotlib.pyplot.
cm = confusion_matrix(y_test, y_pred)
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(image_dataset.target_names))
plt.xticks(tick_marks, image_dataset.target_names, rotation=45)
plt.yticks(tick_marks, image_dataset.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


##I see that my model performs fairly well overall with 76% accuracy, but the classes behave very differently. Class 0, 2, and 3 are recognized strongly, while class 1 and class 4 remain challenging, especially with class 1 being frequently misclassified as class 4. The confusion matrix confirms these patterns and helps me pinpoint exactly where my model still needs improvement.

In [ ]:
#Feature Importance Visualization:
plt.figure(figsize=(10, 6))
plt.barh(range(len(rf_clf.feature_importances_)), rf_clf.feature_importances_)
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Random Forest Feature Importance')
plt.show()

#Get feature importances from the best model using the feature_importances_ attribute.
feature_importances = rf_clf.feature_importances_


#Create a bar plot of feature importances using matplotlib.pyplot.
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importances)), feature_importances)
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Random Forest Feature Importance')
plt.show()

In [ ]:
#Prediction on New Images:
#Implement a function to predict the class of a new image.
def predict_image(image_path, model, image_dataset):
    img = imread(image_path)
    img_resized = resize(img, (64, 64), anti_aliasing=True, mode='reflect')
    flat_data = img_resized.flatten()
    flat_data = np.array(flat_data)
    flat_data = flat_data.reshape(1, -1)
    prediction = model.predict(flat_data)
    return image_dataset.target_names[prediction[0]]

#Use the same preprocessing steps as in the data loading function.
x_train, x_test, y_train, y_test = train_test_split(image_dataset.data, image_dataset.target, test_size=0.3, random_state=42)

#Test the function with a new image and print the predicted class.
predict_image('path/to/new_image.jpg', rf_clf, image_dataset)


In [ ]:
#Comparing with SVM:
#Implement SVM classification using SVC from sklearn.svm.
svm_clf = SVC()
svm_clf.fit(x_train, y_train)

#Follow a similar process as with Random Forest to train and evaluate the SVM model.
y_pred = svm_clf.predict(x_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


#Compare the results of Random Forest and SVM models.
